# Football Manager: data cleaning
Reads the raw ESPN NFL CSVs in this folder, cleans each one and writes the results to `clean/`.

Rules applied to every file:
- Column names become snake_case (Delta-safe: no spaces, dots or brackets)
- Whitespace is trimmed from text and empty strings become null
- Columns that are 100% empty are dropped
- Exact duplicate rows are dropped, and primary keys are checked for uniqueness

`athletes_2026.csv` is byte-for-byte identical to `players.csv`, so only `players.csv` is used.

In [1]:
import ast
import re
from pathlib import Path

import pandas as pd

RAW = Path(".")
OUT = Path("clean")
OUT.mkdir(exist_ok=True)

## Helpers

In [2]:
def to_snake(name: str) -> str:
    """'birthPlace.city' -> 'birth_place_city', 'vs. Div.' -> 'vs_div', 'homeAway' -> 'home_away'."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return re.sub(r"_+", "_", name).strip("_").lower()


def base_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=to_snake)
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].str.strip().replace("", pd.NA)
    df = df.dropna(axis=1, how="all")
    return df.drop_duplicates().reset_index(drop=True)


def to_int(df: pd.DataFrame, cols) -> pd.DataFrame:
    """Nullable integer (Int64) so NaN doesn't force floats like 210.0."""
    for c in cols:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")
    return df


def to_bool(df: pd.DataFrame, cols) -> pd.DataFrame:
    m = {"true": True, "false": False}
    for c in cols:
        if c in df:
            df[c] = df[c].astype(str).str.lower().map(m).astype("boolean")
    return df


def split_pair(df: pd.DataFrame, col: str, left: str, right: str, sep: str = "-") -> pd.DataFrame:
    """'5-8' -> two int columns. Drops the original."""
    parts = df[col].str.split(sep, n=1, expand=True)
    df[left] = pd.to_numeric(parts[0], errors="coerce").astype("Int64")
    df[right] = pd.to_numeric(parts[1], errors="coerce").astype("Int64")
    return df.drop(columns=col)


def check_pk(df: pd.DataFrame, keys, name: str) -> None:
    dups = df.duplicated(keys).sum()
    nulls = df[keys].isna().any(axis=1).sum()
    assert dups == 0 and nulls == 0, f"{name}: {dups} duplicate / {nulls} null keys on {keys}"


def save(df: pd.DataFrame, name: str) -> None:
    df.to_csv(OUT / f"{name}.csv", index=False)
    print(f"{name:<16} {df.shape[0]:>5} rows x {df.shape[1]:>2} cols -> {OUT / (name + '.csv')}")

## teams

In [3]:
teams = base_clean(pd.read_csv(RAW / "teams.csv"))
teams = to_int(teams, ["team_id"])
teams = to_bool(teams, ["is_active"])
teams = teams.rename(columns={"display_name": "team_name", "short_display_name": "short_name"})
for c in ["color", "alternate_color"]:
    teams[c] = "#" + teams[c].str.lower()
check_pk(teams, ["team_id"], "teams")
save(teams, "teams")

teams               32 rows x 11 cols -> clean/teams.csv


## players
Drops the `display_*` columns (they only repeat weight/height as text) and the
`hand_abbreviation` / `hand_display_value` columns (same info as `hand_type`).

In [4]:
players = base_clean(pd.read_csv(RAW / "players.csv"))
players = players.rename(columns={"id": "player_id"})
players = players.drop(columns=["display_weight", "display_height", "hand_abbreviation", "hand_display_value"], errors="ignore")
players = to_int(players, ["player_id", "weight", "height", "age", "jersey", "experience_years"])
players = to_bool(players, ["active"])
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], utc=True, errors="coerce").dt.date
players = players.rename(columns={"weight": "weight_lbs", "height": "height_in"})
if "hand_type" in players:
    players["hand_type"] = players["hand_type"].str.lower()
check_pk(players, ["player_id"], "players")
save(players, "players")

players           4059 rows x 22 cols -> clean/players.csv


## rosters

In [5]:
rosters = base_clean(pd.read_csv(RAW / "rosters.csv"))
rosters = to_int(rosters, ["season", "team_id", "player_id", "jersey", "age", "height", "weight", "experience_years"])
rosters = rosters.rename(columns={"full_name": "player_name", "weight": "weight_lbs", "height": "height_in"})
check_pk(rosters, ["season", "team_id", "player_id"], "rosters")
save(rosters, "rosters")

rosters           2485 rows x 12 cols -> clean/rosters.csv


## schedules
Scores arrive as a Python dict string: `"{'value': 21.0, 'displayValue': '21'}"`.
The numeric `value` is extracted. Unplayed games carry a placeholder 0, so scores
are set to null unless `status == 'Final'`.

In [6]:
def parse_score(v):
    if pd.isna(v):
        return pd.NA
    try:
        return ast.literal_eval(v).get("value")
    except (ValueError, SyntaxError):
        return pd.to_numeric(v, errors="coerce")


schedules = base_clean(pd.read_csv(RAW / "schedules.csv"))
for c in ["home_score", "away_score"]:
    schedules[c] = schedules[c].map(parse_score)
schedules = to_int(schedules, ["season", "game_id", "week", "home_team_id", "away_team_id", "home_score", "away_score"])
schedules = to_bool(schedules, ["completed"])
schedules["game_ts_utc"] = pd.to_datetime(schedules.pop("date"), utc=True, errors="coerce")
not_final = schedules["status"] != "Final"
schedules.loc[not_final, ["home_score", "away_score"]] = pd.NA
check_pk(schedules, ["game_id"], "schedules")
save(schedules, "schedules")

schedules         1360 rows x 13 cols -> clean/schedules.csv


## games
`attendance` is empty in the source, so `base_clean` drops it.

In [7]:
games = base_clean(pd.read_csv(RAW / "games.csv"))
games = to_int(games, ["game_id"])
games["game_ts_utc"] = pd.to_datetime(games.pop("date"), utc=True, errors="coerce")
check_pk(games, ["game_id"], "games")
save(games, "games")

games             1360 rows x  3 cols -> clean/games.csv


## game_team_stats
- Combined text stats are split into numeric columns, e.g. `"5-8"` becomes
  `third_down_conversions=5, third_down_attempts=8`
- `possession_time` `"34:42"` becomes `possession_seconds=2082`
- Rows for games not played yet (every stat null) are dropped
- The `*_per_game*` columns are season-to-date averages, not stats for a single game.
  They're dropped here and belong in a team-season table.

In [8]:
gts = base_clean(pd.read_csv(RAW / "game_team_stats.csv", dtype={"thirdDownEff": str}))
per_game_cols = [c for c in gts.columns if "per_game" in c]
gts = gts.drop(columns=per_game_cols)

key_cols = ["game_id", "team_id", "team_name", "home_away"]
stat_cols = [c for c in gts.columns if c not in key_cols]
gts = gts.dropna(subset=stat_cols, how="all")

gts = split_pair(gts, "third_down_eff", "third_down_conversions", "third_down_attempts")
gts = split_pair(gts, "fourth_down_eff", "fourth_down_conversions", "fourth_down_attempts")
gts = split_pair(gts, "completion_attempts", "pass_completions", "pass_attempts", sep="/")
gts = split_pair(gts, "sacks_yards_lost", "sacks", "sack_yards_lost")
gts = split_pair(gts, "red_zone_attempts", "red_zone_scores", "red_zone_attempts_total")
gts = gts.rename(columns={"red_zone_attempts_total": "red_zone_attempts"})
gts = split_pair(gts, "total_penalties_yards", "penalties", "penalty_yards")

mmss = gts.pop("possession_time").str.split(":", expand=True)
gts["possession_seconds"] = (pd.to_numeric(mmss[0]) * 60 + pd.to_numeric(mmss[1])).astype("Int64")

int_cols = [c for c in gts.columns if c not in ("team_name", "home_away", "yards_per_play", "yards_per_pass", "yards_per_rush_attempt")]
gts = to_int(gts, int_cols)
check_pk(gts, ["game_id", "team_id"], "game_team_stats")
save(gts, "game_team_stats")

game_team_stats   2208 rows x 34 cols -> clean/game_team_stats.csv


## standings
`group` is identical to `conference`, so it's dropped. The record columns
(`overall`, `home`, `road`, `vs_div`, `vs_conf`) are empty in the source and get dropped.

In [9]:
standings = base_clean(pd.read_csv(RAW / "standings.csv"))
if (standings["group"] == standings["conference"]).all():
    standings = standings.drop(columns="group")
int_cols = ["season", "team_id", "differential", "losses", "playoff_seed", "point_differential",
            "points_against", "points_for", "streak", "ties", "wins", "division_losses",
            "division_ties", "division_wins", "locked_div_rank"]
standings = to_int(standings, int_cols)
standings["clincher"] = standings["clincher"].fillna(0).astype(int).astype(bool)
standings = to_int(standings, ["division_record"])
check_pk(standings, ["season", "team_id"], "standings")
save(standings, "standings")

standings          160 rows x 21 cols -> clean/standings.csv


## Referential integrity checks
Warns (without failing) about foreign keys that don't match a parent table.

In [10]:
def fk_check(child, col, parent, pcol, label):
    missing = set(child[col].dropna()) - set(parent[pcol])
    print(f"{'OK ' if not missing else 'WARN'} {label}: {len(missing)} unmatched" + (f" e.g. {sorted(missing)[:5]}" if missing else ""))

fk_check(rosters, "team_id", teams, "team_id", "rosters.team_id -> teams")
fk_check(rosters, "player_id", players, "player_id", "rosters.player_id -> players")
fk_check(schedules, "home_team_id", teams, "team_id", "schedules.home_team_id -> teams")
fk_check(schedules, "away_team_id", teams, "team_id", "schedules.away_team_id -> teams")
fk_check(gts, "game_id", schedules, "game_id", "game_team_stats.game_id -> schedules")
fk_check(gts, "team_id", teams, "team_id", "game_team_stats.team_id -> teams")
fk_check(games, "game_id", schedules, "game_id", "games.game_id -> schedules")
fk_check(standings, "team_id", teams, "team_id", "standings.team_id -> teams")

OK  rosters.team_id -> teams: 0 unmatched
WARN rosters.player_id -> players: 759 unmatched e.g. [np.int64(11394), np.int64(2094390), np.int64(2310331), np.int64(2512477), np.int64(2574891)]
OK  schedules.home_team_id -> teams: 0 unmatched
OK  schedules.away_team_id -> teams: 0 unmatched
OK  game_team_stats.game_id -> schedules: 0 unmatched
OK  game_team_stats.team_id -> teams: 0 unmatched
OK  games.game_id -> schedules: 0 unmatched
OK  standings.team_id -> teams: 0 unmatched
